In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="12eC1zL3eWd5rs5cJw-xmsmbMZUU4JumZ", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why Rag Analogy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_rag_analogy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# Why RAG? The Retrieval Revolution

*Part 1 of the Vizuara series on RAG Systems*
*Estimated time: 45 minutes*

Let us start with an analogy. Imagine you have a brilliant colleague — someone who has read thousands of textbooks, research papers, and articles. They are incredibly knowledgeable. But there is a catch: **they stopped reading anything new on January 1st, 2024.** Since that day, they have not opened a single newspaper, journal, or website.

You ask them: "What was the biggest AI breakthrough this month?" They stare at you blankly. Not because they are unintelligent — their reasoning is exceptional — but because the answer simply is not in their memory. Their knowledge is **frozen**.

This is precisely the problem with Large Language Models. An LLM's knowledge is baked into its weights during training — what researchers call **parametric knowledge**. It has a cutoff date. It cannot access your company's internal documentation. It does not know what happened yesterday.

But what if we could hand our brilliant colleague a stack of relevant documents *before* they answer each question? Suddenly, their frozen knowledge does not matter as much — they combine their deep understanding with fresh, relevant information to produce an accurate, grounded answer.

This is the core idea behind **Retrieval-Augmented Generation (RAG)**, and by the end of this notebook, you will have built a complete minimal RAG system from scratch.

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/rag-systems/practice/1/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Code Walkthrough: Setup Libs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_setup_libs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Setup

Let us install the libraries we need. We use `sentence-transformers` for embedding text and `faiss-cpu` for fast similarity search — these are **tools**, not the concepts we are learning. The concept is the retrieval pipeline itself.

In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy matplotlib pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import textwrap
import time
from typing import List, Tuple, Dict, Optional

# Reproducibility
np.random.seed(42)

# Plotting defaults — clean, publication-quality figures
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

print("Setup complete! Let us begin.")

In [ ]:
#@title 🎧 Listen: Intuition Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_intuition_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building Intuition — Why Retrieval Changes Everything

Before we write any code or look at any equations, let us build intuition for what RAG actually does.

### The Phone Call Analogy

Imagine you are taking a phone call from a customer who asks: "What is the return policy for items purchased during the holiday sale?"

**Approach 1 — Parametric Only (No Retrieval):**
You try to answer from memory. You vaguely recall the return policy is 30 days, but was there a special extension for holiday purchases? You are not sure. You give your best guess: "I believe it is 30 days." The customer is not satisfied — they need a definitive answer.

**Approach 2 — Retrieval-Augmented:**
Before answering, you pull up the company's return policy document on your screen. You scan it, find the relevant section: "Items purchased during the holiday sale (Nov 24 - Dec 31) have a 60-day return window." Now you answer with confidence and cite the source: "According to our holiday policy, you have 60 days. I am looking at the document right now."

The difference is night and day. Your intelligence did not change between the two approaches — what changed is **what information you had access to** at the moment of answering.

This is exactly what RAG does for LLMs:

| Aspect | Parametric Only | Retrieval-Augmented |
|--------|----------------|-------------------|
| **Knowledge** | Frozen at training time | Fresh, up-to-date documents |
| **Accuracy** | Depends on memorization | Grounded in retrieved evidence |
| **Verifiability** | No sources to check | Citations point to exact documents |
| **Domain coverage** | Only what was in training data | Any knowledge base you connect |
| **Hallucination risk** | High for rare or recent facts | Much lower — constrained by evidence |

### The Verifiability Benefit

This last point — verifiability — deserves special attention. In high-stakes domains like medicine, law, and finance, an answer without a source is nearly useless. A doctor does not want an AI that says "take this medication." They want an AI that says "based on the 2024 clinical guidelines (Section 3.2), this medication is recommended for patients with your profile." RAG makes this possible by attaching source citations to every answer.

In [ ]:
#@title 🎧 Listen: Think About This
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_think_about_this.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Think About This

Before we dive into the mathematics, consider these questions:

- Why can we not simply retrain the LLM every time new information becomes available? (Hint: training GPT-4 cost over $100 million and took months.)
- Why is retrieval better than just stuffing the entire knowledge base into the prompt? (Hint: context windows have limits, and irrelevant information actually *hurts* performance.)
- What kinds of questions would RAG help with the most? What kinds would it help with the least?

Take a moment to think about these. Then let us look at the mathematics.

In [ ]:
#@title 🎧 Listen: Math Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_math_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. The Mathematics — How Retrieval Improves Generation

### The Core Formulation

Let us formalize what we just discussed intuitively. In a standard LLM, the model computes the probability of an answer $y$ given a question $x$:

$$P(y \mid x) = P_{\text{parametric}}(y \mid x)$$

This is the parametric-only approach — the model relies entirely on what it memorized during training.

In RAG, we introduce a **retrieval step**. We first retrieve a set of relevant documents $z$ from a knowledge base, then generate the answer conditioned on *both* the question and the retrieved documents:

$$P(y \mid x) = \sum_{z \in \mathcal{Z}} P(y \mid x, z) \cdot P(z \mid x)$$

What does this equation say in plain English?

- **$P(z \mid x)$** — The retriever's job: given the question $x$, how relevant is document $z$? This is the probability that we would retrieve document $z$ for this question.
- **$P(y \mid x, z)$** — The generator's job: given the question $x$ AND the retrieved document $z$, what is the probability of generating answer $y$?
- **The sum** — We consider all possible retrieved documents and weight each answer by how relevant its supporting document was.

In practice, we do not sum over *all* documents. We retrieve the top-$k$ most relevant ones and use those.

### Worked Numerical Example

Let us plug in concrete numbers to see how retrieval improves accuracy. Suppose we have a factual question, and we measure the probability of the model generating the correct answer.

**Without retrieval (parametric only):**
The model has a 30% chance of producing the correct answer from memory alone:

$$P_{\text{parametric}}(y_{\text{correct}} \mid x) = 0.30$$

**With retrieval:**
Our retriever finds the right document 90% of the time:

$$P(z_{\text{relevant}} \mid x) = 0.90$$

And when the model has the right document, it generates the correct answer 95% of the time:

$$P(y_{\text{correct}} \mid x, z_{\text{relevant}}) = 0.95$$

So the RAG accuracy becomes:

$$P_{\text{RAG}}(y_{\text{correct}} \mid x) = P(y \mid x, z) \cdot P(z \mid x) = 0.95 \times 0.90 = 0.855$$

We went from **30% accuracy to 85.5% accuracy** — nearly a 3x improvement — without changing the model at all. We just gave it better information at inference time.

This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Worked Example
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_worked_example.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Let us verify this calculation and extend it across different question types.

In [ ]:
#@title 🎧 Code Walkthrough: Accuracy Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_accuracy_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Worked Example: Parametric vs RAG Accuracy ---

# Define accuracy components for 5 different question types
question_types = [
    "Recent Events",
    "Domain-Specific\n(e.g., Medical)",
    "Common\nKnowledge",
    "Company\nInternal Docs",
    "Numerical\nFacts",
]

# Parametric-only accuracy (from memory alone)
parametric_accuracy = np.array([0.05, 0.40, 0.85, 0.00, 0.25])

# RAG components: retrieval precision and generation accuracy given good context
retrieval_precision = np.array([0.85, 0.90, 0.70, 0.95, 0.88])
generation_accuracy = np.array([0.90, 0.92, 0.95, 0.90, 0.93])

# RAG accuracy = P(y|x,z) * P(z|x)
rag_accuracy = retrieval_precision * generation_accuracy

print("Accuracy Comparison: Parametric-Only vs RAG")
print("=" * 70)
print(f"{'Question Type':<25} {'Parametric':>12} {'Retrieval P(z|x)':>16} {'Gen P(y|x,z)':>14} {'RAG':>8}")
print("-" * 70)

for i, qtype in enumerate(question_types):
    qtype_clean = qtype.replace('\n', ' ')
    print(f"{qtype_clean:<25} {parametric_accuracy[i]:>11.0%} {retrieval_precision[i]:>15.0%} "
          f"{generation_accuracy[i]:>13.0%} {rag_accuracy[i]:>7.1%}")

print("-" * 70)
print(f"{'Average':<25} {parametric_accuracy.mean():>11.1%} {retrieval_precision.mean():>15.1%} "
      f"{generation_accuracy.mean():>13.1%} {rag_accuracy.mean():>7.1%}")

In [ ]:
#@title 🎧 What to Look For: Accuracy Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_accuracy_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualization: Parametric-Only vs RAG Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(question_types))
width = 0.35

# Bar chart comparing parametric vs RAG
bars_param = ax.bar(x - width/2, parametric_accuracy * 100, width,
                     label='Parametric Only', color='#EF4444', alpha=0.85,
                     edgecolor='white', linewidth=1.5)
bars_rag = ax.bar(x + width/2, rag_accuracy * 100, width,
                   label='RAG (Retrieve + Generate)', color='#3B82F6', alpha=0.85,
                   edgecolor='white', linewidth=1.5)

# Add value labels on each bar
for bar in bars_param:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1.5,
            f'{height:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='#EF4444')

for bar in bars_rag:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1.5,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold',
            color='#3B82F6')

# Add improvement arrows between bars
for i in range(len(question_types)):
    improvement = rag_accuracy[i] * 100 - parametric_accuracy[i] * 100
    if improvement > 5:
        mid_x = x[i]
        mid_y = max(parametric_accuracy[i], rag_accuracy[i]) * 100 + 8
        ax.annotate(f'+{improvement:.0f}%',
                    xy=(mid_x, mid_y), fontsize=9, fontweight='bold',
                    ha='center', color='#059669',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='#D1FAE5',
                              edgecolor='#059669', alpha=0.9))

ax.set_xlabel('Question Type', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('The RAG Advantage: Accuracy Across Question Types\n'
             r'RAG accuracy = $P(y|x,z) \times P(z|x)$',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(question_types, fontsize=10)
ax.set_ylim(0, 110)
ax.legend(fontsize=12, loc='upper left')

plt.tight_layout()
plt.show()

print("\nKey insight: RAG provides the largest improvement where parametric knowledge")
print("is weakest — recent events, domain-specific facts, and internal documents.")
print("For common knowledge, the LLM already does well, so RAG helps less.")

In [ ]:
#@title 🎧 Transition: Build It Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_build_it_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Let Us Build It — A Minimal RAG System from Scratch

Now we build a complete RAG system step by step. Each component is self-contained and testable.

### 4.1 The Knowledge Base

First, let us create our knowledge base — a collection of text passages about renewable energy and climate science. We deliberately choose a factual domain where accuracy and citations matter.

In [ ]:
#@title 🎧 Code Walkthrough: Knowledge Base
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_knowledge_base.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# --- Knowledge Base: 12 passages about renewable energy and climate ---
knowledge_base = [
    # Solar energy
    "Solar photovoltaic (PV) cells convert sunlight directly into electricity using the "
    "photovoltaic effect. When photons strike a semiconductor material like silicon, they "
    "knock electrons free, creating an electric current. Modern commercial solar panels "
    "achieve 20-22% efficiency, while laboratory cells have reached 47% efficiency using "
    "multi-junction designs.",

    "The cost of solar energy has dropped by 89% since 2010, making it the cheapest source "
    "of new electricity generation in most of the world. In 2023, global solar capacity "
    "reached 1.6 terawatts, with China accounting for over 50% of new installations.",

    "Solar panel degradation occurs at roughly 0.5% per year, meaning a panel installed "
    "today will still produce about 87.5% of its original output after 25 years. This "
    "longevity makes solar one of the most cost-effective long-term energy investments.",

    # Wind energy
    "Modern wind turbines convert kinetic energy from wind into electricity using large "
    "rotor blades connected to a generator. The theoretical maximum efficiency of a wind "
    "turbine is 59.3%, known as the Betz limit. Modern turbines achieve 35-45% efficiency "
    "in practice.",

    "Offshore wind farms produce 50-70% more energy than onshore installations due to "
    "stronger and more consistent wind speeds over water. The average offshore wind turbine "
    "has a capacity of 8-15 megawatts, compared to 2-5 megawatts for onshore turbines.",

    # Energy storage
    "Lithium-ion batteries dominate the energy storage market with over 90% market share. "
    "Battery costs have fallen from $1,100 per kilowatt-hour in 2010 to approximately "
    "$139 per kilowatt-hour in 2023, a decline of 87%. Grid-scale battery installations "
    "reached 45 gigawatt-hours globally in 2023.",

    "Pumped hydro storage accounts for over 90% of global energy storage capacity. It works "
    "by pumping water uphill to a reservoir when electricity is cheap, then releasing it "
    "through turbines to generate power during peak demand. Round-trip efficiency is 70-85%.",

    # Climate science
    "The global average temperature has risen by approximately 1.1 degrees Celsius above "
    "pre-industrial levels as of 2023. The Paris Agreement aims to limit warming to 1.5 "
    "degrees Celsius, requiring global emissions to reach net zero by 2050.",

    "Carbon dioxide concentrations in the atmosphere reached 421 parts per million (ppm) "
    "in 2023, the highest level in at least 800,000 years. Pre-industrial levels were "
    "approximately 280 ppm. Each ppm increase corresponds to roughly 7.8 gigatons of "
    "carbon added to the atmosphere.",

    # Grid and policy
    "The duck curve describes the shape of the electricity demand chart in regions with "
    "high solar penetration. During midday, solar generation exceeds demand, causing net "
    "load to drop sharply. As the sun sets, conventional generators must ramp up quickly "
    "to meet evening demand, creating grid stability challenges.",

    "Green hydrogen is produced by splitting water into hydrogen and oxygen using "
    "electrolysis powered by renewable electricity. It costs $3-6 per kilogram to produce "
    "as of 2023, compared to $1-2 per kilogram for gray hydrogen made from natural gas. "
    "Cost parity is expected by 2030 in favorable regions.",

    "The capacity factor of an energy source measures actual output as a percentage of "
    "maximum possible output. Nuclear power has the highest capacity factor at 92%, "
    "followed by natural gas at 57%, wind at 34%, and solar at 25%. This means a 1 GW "
    "solar farm produces roughly the same annual energy as a 270 MW nuclear plant.",
]

print(f"Knowledge base created: {len(knowledge_base)} passages")
print(f"\nTopics covered:")
topics = ["Solar Energy (3 passages)", "Wind Energy (2 passages)",
          "Energy Storage (2 passages)", "Climate Science (2 passages)",
          "Grid & Policy (3 passages)"]
for topic in topics:
    print(f"  - {topic}")

print(f"\nExample passage (first 120 chars):")
print(f'  "{knowledge_base[0][:120]}..."')

In [ ]:
#@title 🎧 Listen: Text Chunking
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_text_chunking.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.2 Text Chunking — Breaking Documents Into Retrievable Pieces

In a real RAG system, documents can be thousands of words long. We break them into smaller **chunks** so that we can retrieve only the most relevant portion rather than the entire document.

We implement two strategies: fixed-size chunking and fixed-size with overlap. Later, you will compare them.

In [ ]:
def chunk_fixed_size(text: str, chunk_size: int = 200, overlap: int = 0) -> List[str]:
    """
    Split text into chunks of approximately `chunk_size` characters.

    Args:
        text: The input text to chunk
        chunk_size: Target size of each chunk in characters
        overlap: Number of overlapping characters between consecutive chunks

    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        # Slide window forward by (chunk_size - overlap)
        step = max(chunk_size - overlap, 1)
        start += step
    return chunks


# Since our knowledge base passages are already reasonable length,
# let us chunk only the longer ones for demonstration
print("Chunking demonstration on a longer passage:")
print("-" * 60)

long_passage = knowledge_base[0]  # Solar PV passage
print(f"Original ({len(long_passage)} chars):")
print(f'  "{long_passage[:100]}..."')

# Chunk without overlap
chunks_no_overlap = chunk_fixed_size(long_passage, chunk_size=150, overlap=0)
print(f"\nChunked (150 chars, no overlap) -> {len(chunks_no_overlap)} chunks:")
for i, chunk in enumerate(chunks_no_overlap):
    print(f'  Chunk {i+1} ({len(chunk)} chars): "{chunk[:60]}..."')

# Chunk with overlap
chunks_with_overlap = chunk_fixed_size(long_passage, chunk_size=150, overlap=50)
print(f"\nChunked (150 chars, 50 overlap) -> {len(chunks_with_overlap)} chunks:")
for i, chunk in enumerate(chunks_with_overlap):
    print(f'  Chunk {i+1} ({len(chunk)} chars): "{chunk[:60]}..."')

print("\nNotice: overlap creates more chunks but preserves context at boundaries.")

For this notebook, our passages are already a good size for retrieval (each is roughly 200-400 characters). In a production system, you would be chunking much longer documents — PDFs, web pages, entire book chapters. The chunking strategy matters enormously there. For now, let us use our passages as-is and move on to the exciting part: embedding.


In [ ]:
#@title 🎧 Listen: Embedding Chunks
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_embedding_chunks.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.3 Embedding Chunks with Sentence-Transformers

Now we convert text into vectors — high-dimensional numerical representations that capture semantic meaning. Two passages about solar energy will have similar vectors even if they use completely different words.

We use `all-MiniLM-L6-v2`, a small and fast model that produces 384-dimensional embeddings.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("Loading embedding model (first run downloads ~80MB)...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded! Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

# Embed our entire knowledge base
print(f"\nEmbedding {len(knowledge_base)} passages...")
start_time = time.time()
kb_embeddings = embed_model.encode(knowledge_base, show_progress_bar=True)
kb_embeddings = np.array(kb_embeddings)
elapsed = time.time() - start_time

print(f"\nDone in {elapsed:.2f}s")
print(f"Embedding matrix shape: {kb_embeddings.shape}")
print(f"  -> {kb_embeddings.shape[0]} passages, {kb_embeddings.shape[1]} dimensions each")
print(f"  -> Each passage is now a vector of {kb_embeddings.shape[1]} numbers")

Let us peek at what an embedding actually looks like.

In [ ]:
# Inspect one embedding
print(f"Passage: '{knowledge_base[0][:70]}...'")
print(f"\nEmbedding (first 20 of {kb_embeddings.shape[1]} dimensions):")
print(f"  {kb_embeddings[0][:20]}")
print(f"\nEmbedding L2 norm: {np.linalg.norm(kb_embeddings[0]):.4f}")
print("  (sentence-transformers normalizes to approximately unit length)")

In [ ]:
#@title 🎧 Listen: Faiss Index
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_faiss_index.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.4 Building a FAISS Index

FAISS (Facebook AI Similarity Search) is a library built for fast nearest-neighbor search over dense vectors. Instead of comparing our query against every document (which works fine for 12 passages but becomes slow at millions), FAISS uses optimized data structures.

For our small knowledge base, we use a flat (brute-force) index — but the API is the same whether you have 12 documents or 12 million.

In [ ]:
import faiss

# Build a FAISS index
embedding_dim = kb_embeddings.shape[1]  # 384

# FlatIP = Flat Inner Product (cosine similarity when vectors are normalized)
index = faiss.IndexFlatIP(embedding_dim)

# Normalize embeddings for cosine similarity via inner product
faiss.normalize_L2(kb_embeddings)

# Add all document embeddings to the index
index.add(kb_embeddings.astype(np.float32))

print(f"FAISS index built!")
print(f"  Index type: Flat Inner Product (brute-force cosine similarity)")
print(f"  Vectors stored: {index.ntotal}")
print(f"  Dimension: {embedding_dim}")
print(f"\nFor 12 passages, brute-force is fine.")
print(f"For 12 million passages, you would switch to an approximate index (IVF, HNSW).")

In [ ]:
#@title 🎧 Code Walkthrough: Implement Retrieval
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_implement_retrieval.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.5 Implementing Retrieval — Top-K Search

Now for the core operation: given a query, find the most relevant passages.

In [ ]:
def retrieve(query: str, top_k: int = 3) -> List[Tuple[str, float, int]]:
    """
    Retrieve the top-k most relevant passages for a given query.

    Args:
        query: The user's question
        top_k: Number of passages to retrieve

    Returns:
        List of (passage_text, similarity_score, passage_index) tuples,
        sorted by relevance (highest first)
    """
    # Step 1: Embed the query
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding, dtype=np.float32)
    faiss.normalize_L2(query_embedding)

    # Step 2: Search the FAISS index
    scores, indices = index.search(query_embedding, top_k)

    # Step 3: Package results
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append((knowledge_base[idx], float(score), int(idx)))

    return results


# Test retrieval on a few queries
test_queries = [
    "How efficient are solar panels?",
    "What is the cost of battery storage?",
    "How much has the global temperature risen?",
]

print("=" * 70)
print("RETRIEVAL TEST: Top-3 results for each query")
print("=" * 70)

for query in test_queries:
    results = retrieve(query, top_k=3)
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    for rank, (passage, score, idx) in enumerate(results, 1):
        preview = passage[:90] + "..." if len(passage) > 90 else passage
        print(f"  {rank}. [score={score:.4f}] {preview}")

This is exactly what we want. The retriever is finding semantically relevant passages even when the query uses different words than the passage. "How efficient are solar panels?" matches the passage about photovoltaic efficiency, not by keyword overlap, but by meaning.


In [ ]:
#@title 🎧 Code Walkthrough: Prompt Template
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_prompt_template.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### 4.6 Building the Generation Prompt Template

The final piece: once we have retrieved relevant passages, we need to assemble them into a prompt that an LLM can use to generate a grounded answer.

In [ ]:
def build_rag_prompt(query: str, retrieved_passages: List[Tuple[str, float, int]]) -> str:
    """
    Build a structured prompt for the LLM with retrieved context.

    The prompt uses clear section markers so the LLM knows:
    - What its role is (system instruction)
    - What evidence it has (retrieved passages with source IDs)
    - What question to answer (the user query)
    - How to format its response (with citations)
    """
    # Format retrieved passages with source IDs
    context_parts = []
    for i, (passage, score, idx) in enumerate(retrieved_passages, 1):
        context_parts.append(f"[Source {i}] (relevance: {score:.3f})\n{passage}")

    context_str = "\n\n".join(context_parts)

    prompt = f"""You are a knowledgeable assistant that answers questions based on the provided sources.

INSTRUCTIONS:
- Answer the question using ONLY the information in the provided sources.
- Cite your sources using [Source N] notation.
- If the sources do not contain enough information to answer, say so explicitly.
- Be concise but thorough.

SOURCES:
{context_str}

QUESTION: {query}

ANSWER:"""

    return prompt


# Demo: build a prompt
demo_query = "What is the current cost of solar energy and how has it changed?"
demo_results = retrieve(demo_query, top_k=3)
demo_prompt = build_rag_prompt(demo_query, demo_results)

print("=" * 70)
print("ASSEMBLED RAG PROMPT (what the LLM would see)")
print("=" * 70)
print(demo_prompt)
print("=" * 70)
print(f"\nPrompt length: {len(demo_prompt)} characters (~{len(demo_prompt)//4} tokens)")

In [ ]:
#@title 🎧 What to Look For: Tsne Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_tsne_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualization: 2D t-SNE Projection of Embedded Passages

Let us visualize our embedded knowledge base in 2D to build intuition for how semantic similarity works in embedding space. Passages about similar topics should cluster together.

In [ ]:
from sklearn.manifold import TSNE

# Run t-SNE to project 384D embeddings down to 2D
np.random.seed(42)
tsne = TSNE(n_components=2, perplexity=5, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(kb_embeddings)

# Assign colors by topic
topic_labels = [
    "Solar", "Solar", "Solar",           # Passages 0-2
    "Wind", "Wind",                       # Passages 3-4
    "Storage", "Storage",                 # Passages 5-6
    "Climate", "Climate",                 # Passages 7-8
    "Grid/Policy", "Grid/Policy", "Grid/Policy",  # Passages 9-11
]
topic_colors = {
    "Solar": "#F59E0B",
    "Wind": "#3B82F6",
    "Storage": "#10B981",
    "Climate": "#EF4444",
    "Grid/Policy": "#8B5CF6",
}

fig, ax = plt.subplots(figsize=(12, 8))

# Plot each passage as a colored point
for i, (x, y) in enumerate(embeddings_2d):
    topic = topic_labels[i]
    color = topic_colors[topic]
    ax.scatter(x, y, c=color, s=200, alpha=0.8, edgecolors='white', linewidth=1.5, zorder=3)

    # Add passage number label
    ax.annotate(f'P{i}', (x, y), textcoords="offset points", xytext=(8, 5),
                fontsize=9, fontweight='bold', color=color)

    # Add a short text preview
    preview = knowledge_base[i][:45] + "..."
    ax.annotate(preview, (x, y), textcoords="offset points", xytext=(8, -10),
                fontsize=7, color='#666', style='italic')

# Now embed and plot a query to show where it lands
query_text = "How efficient are solar panels?"
query_emb = embed_model.encode([query_text])
query_emb_norm = query_emb.copy()
faiss.normalize_L2(query_emb_norm)

# Project query into 2D (approximate: use nearest neighbor position)
# For visualization, we append query to embeddings and re-run t-SNE
all_embs = np.vstack([kb_embeddings, query_emb_norm])
tsne_all = TSNE(n_components=2, perplexity=5, random_state=42, n_iter=1000)
all_2d = tsne_all.fit_transform(all_embs.astype(np.float32))
query_2d = all_2d[-1]

# Re-plot with query
fig, ax = plt.subplots(figsize=(14, 9))

for i, (x_pt, y_pt) in enumerate(all_2d[:-1]):
    topic = topic_labels[i]
    color = topic_colors[topic]
    ax.scatter(x_pt, y_pt, c=color, s=200, alpha=0.8, edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(f'P{i}', (x_pt, y_pt), textcoords="offset points", xytext=(8, 5),
                fontsize=9, fontweight='bold', color=color)

# Plot the query as a star
ax.scatter(query_2d[0], query_2d[1], c='red', s=400, marker='*', zorder=5,
           edgecolors='darkred', linewidth=1.5)
ax.annotate(f'Query: "{query_text}"', (query_2d[0], query_2d[1]),
            textcoords="offset points", xytext=(12, 12),
            fontsize=10, fontweight='bold', color='darkred',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FEE2E2', edgecolor='#EF4444'))

# Legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [Patch(facecolor=c, label=t, alpha=0.8) for t, c in topic_colors.items()]
legend_elements.append(Line2D([0], [0], marker='*', color='w', markerfacecolor='red',
                               markersize=15, label='Query'))
ax.legend(handles=legend_elements, loc='upper right', fontsize=10, framealpha=0.9)

ax.set_title('t-SNE Projection of Knowledge Base Embeddings\n'
             'Passages about similar topics cluster together',
             fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)

plt.tight_layout()
plt.show()

print("\nNotice how passages about the same topic (same color) cluster together.")
print("The query (red star) lands near the solar energy cluster — exactly what we expect.")
print("This is why embedding-based retrieval works: semantic similarity = geometric proximity.")

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Your Turn — Hands-On Exercises

### TODO 1: Implement a Retrieval Function with Similarity Scores

Your task: implement a function that takes a query, embeds it, searches the FAISS index, and returns the top-k results with detailed similarity information.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def retrieve_with_details(
    query: str,
    knowledge_base: List[str],
    embed_model: SentenceTransformer,
    index: faiss.Index,
    top_k: int = 3,
) -> List[Dict]:
    """
    Retrieve top-k relevant passages with detailed similarity information.

    Args:
        query: The user's question string
        knowledge_base: List of original text passages
        embed_model: The SentenceTransformer model for encoding
        index: The FAISS index containing passage embeddings
        top_k: Number of results to return

    Returns:
        List of dicts, each containing:
            - 'text': the passage text
            - 'score': cosine similarity score (float)
            - 'index': index into the knowledge base (int)
            - 'preview': first 80 characters of the passage (str)
            - 'rank': 1-based rank (int)

    Hints:
        Step 1: Encode the query using embed_model.encode([query])
        Step 2: Convert to float32 numpy array
        Step 3: Normalize with faiss.normalize_L2()
        Step 4: Search with index.search(query_embedding, top_k)
        Step 5: Package each result into a dict with all required fields
    """
    # ============ YOUR CODE HERE ============

    # Step 1: Embed the query
    query_embedding = ???

    # Step 2: Convert to float32
    query_embedding = ???

    # Step 3: Normalize for cosine similarity
    ???

    # Step 4: Search the index
    scores, indices = ???

    # Step 5: Build result list
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        results.append({
            'text': ???,
            'score': ???,
            'index': ???,
            'preview': ???,
            'rank': ???,
        })

    # ========================================

    return results

In [ ]:
#@title 🎧 Before You Start: Todo1 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_todo1_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification: Run this cell to check your implementation

test_result = retrieve_with_details(
    "How efficient are solar panels?",
    knowledge_base, embed_model, index, top_k=3
)

# Check structure
assert isinstance(test_result, list), "Should return a list"
assert len(test_result) == 3, f"Should return 3 results, got {len(test_result)}"
assert all(isinstance(r, dict) for r in test_result), "Each result should be a dict"

# Check required keys
required_keys = {'text', 'score', 'index', 'preview', 'rank'}
for r in test_result:
    assert required_keys.issubset(r.keys()), f"Missing keys: {required_keys - r.keys()}"

# Check that scores are descending
scores = [r['score'] for r in test_result]
assert scores == sorted(scores, reverse=True), "Results should be sorted by score (descending)"

# Check that ranks are 1, 2, 3
ranks = [r['rank'] for r in test_result]
assert ranks == [1, 2, 3], f"Ranks should be [1, 2, 3], got {ranks}"

# Check that top result is about solar energy
assert "solar" in test_result[0]['text'].lower(), \
    "Top result for 'How efficient are solar panels?' should be about solar energy"

# Check preview length
assert len(test_result[0]['preview']) <= 83, "Preview should be at most 80 chars + '...'"

print("All tests passed! Your retrieval function works correctly.")
print(f"\nTop result for 'How efficient are solar panels?':")
print(f"  Rank: {test_result[0]['rank']}")
print(f"  Score: {test_result[0]['score']:.4f}")
print(f"  Preview: {test_result[0]['preview']}")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Compare Chunking Strategies — With vs Without Overlap

Your task: implement overlap-based chunking and compare retrieval quality against non-overlap chunking.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def evaluate_chunking_strategy(
    passages: List[str],
    queries: List[Tuple[str, int]],
    chunk_size: int,
    overlap: int,
    embed_model: SentenceTransformer,
) -> Dict:
    """
    Evaluate a chunking strategy by measuring retrieval quality.

    Args:
        passages: List of text passages to chunk
        queries: List of (query_string, expected_passage_index) tuples.
                 expected_passage_index is the index of the passage that
                 should be retrieved for this query.
        chunk_size: Character size of each chunk
        overlap: Number of overlapping characters between chunks
        embed_model: SentenceTransformer for encoding

    Returns:
        Dict with keys:
            - 'num_chunks': total number of chunks created
            - 'avg_chunk_size': average chunk size in characters
            - 'hits_at_1': number of queries where top-1 chunk came from correct passage
            - 'hits_at_3': number of queries where any top-3 chunk came from correct passage
            - 'accuracy_at_1': hits_at_1 / total queries
            - 'accuracy_at_3': hits_at_3 / total queries

    Hints:
        Step 1: Chunk each passage using chunk_fixed_size(passage, chunk_size, overlap)
        Step 2: Keep track of which passage each chunk came from (chunk_origins list)
        Step 3: Embed all chunks with embed_model.encode()
        Step 4: Build a temporary FAISS index for these chunks
        Step 5: For each query, retrieve top-3 chunks and check if any came from
                the expected passage
    """
    # ============ YOUR CODE HERE ============

    # Step 1: Chunk all passages, track origins
    all_chunks = []
    chunk_origins = []  # chunk_origins[i] = index of the passage that chunk i came from
    for passage_idx, passage in enumerate(passages):
        chunks = ???
        for chunk in chunks:
            all_chunks.append(chunk)
            chunk_origins.append(passage_idx)

    # Step 2: Embed all chunks
    chunk_embeddings = ???

    # Step 3: Build a FAISS index
    dim = chunk_embeddings.shape[1]
    temp_index = faiss.IndexFlatIP(dim)
    faiss.normalize_L2(chunk_embeddings)
    temp_index.add(chunk_embeddings.astype(np.float32))

    # Step 4: Evaluate each query
    hits_at_1 = 0
    hits_at_3 = 0

    for query_str, expected_idx in queries:
        q_emb = ???
        faiss.normalize_L2(q_emb)
        scores, indices = temp_index.search(q_emb, 3)

        # Check if top-1 chunk came from the expected passage
        if ???:
            hits_at_1 += 1

        # Check if any of top-3 chunks came from the expected passage
        if ???:
            hits_at_3 += 1

    # ========================================

    num_queries = len(queries)
    return {
        'num_chunks': len(all_chunks),
        'avg_chunk_size': np.mean([len(c) for c in all_chunks]),
        'hits_at_1': hits_at_1,
        'hits_at_3': hits_at_3,
        'accuracy_at_1': hits_at_1 / num_queries,
        'accuracy_at_3': hits_at_3 / num_queries,
    }

In [ ]:
#@title 🎧 Before You Start: Todo2 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_todo2_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Verification: compare overlap vs no-overlap chunking

eval_queries = [
    ("How do solar panels convert sunlight to electricity?", 0),
    ("What is the cost trend of solar energy?", 1),
    ("How long do solar panels last before degrading?", 2),
    ("What is the Betz limit for wind turbines?", 3),
    ("Why is offshore wind better than onshore?", 4),
    ("How much have battery costs declined?", 5),
    ("How does pumped hydro storage work?", 6),
    ("What is the Paris Agreement temperature target?", 7),
    ("What are current CO2 levels in the atmosphere?", 8),
    ("What is the duck curve in electricity?", 9),
]

# Strategy 1: No overlap
result_no_overlap = evaluate_chunking_strategy(
    knowledge_base, eval_queries,
    chunk_size=150, overlap=0, embed_model=embed_model
)

# Strategy 2: With overlap (50 chars)
result_with_overlap = evaluate_chunking_strategy(
    knowledge_base, eval_queries,
    chunk_size=150, overlap=50, embed_model=embed_model
)

print("Chunking Strategy Comparison")
print("=" * 60)
print(f"{'Metric':<25} {'No Overlap':>15} {'50-char Overlap':>15}")
print("-" * 60)
print(f"{'Chunks created':<25} {result_no_overlap['num_chunks']:>15} {result_with_overlap['num_chunks']:>15}")
print(f"{'Avg chunk size':<25} {result_no_overlap['avg_chunk_size']:>14.0f}c {result_with_overlap['avg_chunk_size']:>14.0f}c")
print(f"{'Accuracy @1':<25} {result_no_overlap['accuracy_at_1']:>14.0%} {result_with_overlap['accuracy_at_1']:>14.0%}")
print(f"{'Accuracy @3':<25} {result_no_overlap['accuracy_at_3']:>14.0%} {result_with_overlap['accuracy_at_3']:>14.0%}")
print()

if result_with_overlap['accuracy_at_3'] >= result_no_overlap['accuracy_at_3']:
    print("Overlap helps! Chunks that share boundary context are easier to match.")
else:
    print("Interesting — overlap did not help here. This can happen with short passages.")
print("In production with longer documents, overlap almost always improves quality.")

In [ ]:
#@title 🎧 Transition: Pipeline Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_pipeline_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Putting It All Together — The Complete RAG Pipeline

Now let us wire everything together into a single function that takes a question and returns a formatted answer with source citations.

In [ ]:
#@title 🎧 Code Walkthrough: Pipeline Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_pipeline_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def rag_pipeline(
    query: str,
    top_k: int = 3,
    verbose: bool = True,
) -> Dict:
    """
    Complete RAG pipeline: query -> retrieve -> build prompt -> return result.

    Returns a dict with:
        - query: the original question
        - retrieved: list of (text, score, index) tuples
        - prompt: the assembled prompt string
        - num_sources: how many sources were included
        - total_context_chars: character count of the full prompt
    """
    # Stage 1: Retrieve relevant passages
    retrieved = retrieve(query, top_k=top_k)

    # Stage 2: Build the generation prompt
    prompt = build_rag_prompt(query, retrieved)

    result = {
        'query': query,
        'retrieved': retrieved,
        'prompt': prompt,
        'num_sources': len(retrieved),
        'total_context_chars': len(prompt),
    }

    if verbose:
        print(f"\n{'='*70}")
        print(f"RAG Pipeline Result")
        print(f"{'='*70}")
        print(f"Query: \"{query}\"")
        print(f"\nRetrieved {len(retrieved)} sources:")
        for rank, (text, score, idx) in enumerate(retrieved, 1):
            preview = text[:80] + "..." if len(text) > 80 else text
            print(f"  [{rank}] (score={score:.4f}, passage {idx}) {preview}")
        print(f"\nPrompt size: {len(prompt)} characters (~{len(prompt)//4} tokens)")

    return result


# Run on 3 example queries
example_queries = [
    "What is the current state of solar energy costs?",
    "How does energy storage work at grid scale?",
    "What are the biggest challenges with renewable energy on the grid?",
]

all_results = []
for query in example_queries:
    result = rag_pipeline(query)
    all_results.append(result)
    print()

Let us also see the complete prompt for one of these queries. This is the actual text that would be sent to an LLM like Claude.

In [ ]:
# Show the full assembled prompt for the first query
print("=" * 70)
print("COMPLETE RAG PROMPT (sent to the LLM)")
print("=" * 70)
print(all_results[0]['prompt'])
print("=" * 70)

In [ ]:
#@title 🎧 Listen: Calling Llm
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_calling_llm.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Calling the LLM (Optional — Requires API Key)

If you have an Anthropic API key, uncomment and run this cell to see the full end-to-end RAG pipeline in action — from query to generated answer with citations.

In [ ]:
# Uncomment the following to run with Claude:

# from anthropic import Anthropic
# client = Anthropic()
#
# query = "What is the current state of solar energy and how has its cost changed?"
# result = rag_pipeline(query, top_k=3, verbose=True)
#
# response = client.messages.create(
#     model="claude-sonnet-4-20250514",
#     max_tokens=500,
#     messages=[{"role": "user", "content": result['prompt']}],
# )
#
# print("\n" + "=" * 70)
# print("LLM RESPONSE (grounded in retrieved sources)")
# print("=" * 70)
# print(response.content[0].text)

print("To run the full pipeline with Claude, set your ANTHROPIC_API_KEY")
print("environment variable and uncomment the code above.")

In [ ]:
#@title 🎧 Listen: Metrics Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_metrics_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Measuring Retrieval Quality

Building a RAG system is only half the job. We also need to **measure** how well it retrieves. Two standard metrics for this are **Recall@K** and **Mean Reciprocal Rank (MRR)**.

### Recall@K

Recall@K measures: "Of the relevant documents that exist, how many did we find in our top-K results?"

$$\text{Recall@K} = \frac{|\text{Retrieved@K} \cap \text{Relevant}|}{|\text{Relevant}|}$$

If there are 2 relevant passages and we retrieve both of them in our top-5, then Recall@5 = 2/2 = 1.0. If we only find one, Recall@5 = 1/2 = 0.5.

### Mean Reciprocal Rank (MRR)

MRR measures how high the first relevant result appears in our ranked list:

$$\text{MRR} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

Where $\text{rank}_i$ is the position of the first relevant document for query $i$. If the first relevant document is always at rank 1, MRR = 1.0. If it is always at rank 3, MRR = 1/3.

Let us implement and evaluate both metrics.

In [ ]:
#@title 🎧 Code Walkthrough: Eval Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_28_eval_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def evaluate_retrieval(
    queries_with_ground_truth: List[Tuple[str, set]],
    max_k: int = 10,
) -> Dict:
    """
    Evaluate retrieval quality across a test set.

    Args:
        queries_with_ground_truth: List of (query, set_of_relevant_indices)
        max_k: Maximum K to evaluate

    Returns:
        Dict with recall_at_k (list), mrr (float), per_query results
    """
    recall_at_k = {k: [] for k in range(1, max_k + 1)}
    reciprocal_ranks = []

    per_query_results = []

    for query, relevant_indices in queries_with_ground_truth:
        # Retrieve top-max_k results
        results = retrieve(query, top_k=max_k)
        retrieved_indices = [idx for _, _, idx in results]

        # Compute Recall@K for each K
        for k in range(1, max_k + 1):
            top_k_indices = set(retrieved_indices[:k])
            hits = len(top_k_indices & relevant_indices)
            recall = hits / len(relevant_indices)
            recall_at_k[k].append(recall)

        # Compute Reciprocal Rank
        rr = 0.0
        for rank, idx in enumerate(retrieved_indices, 1):
            if idx in relevant_indices:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

        per_query_results.append({
            'query': query,
            'relevant': relevant_indices,
            'retrieved': retrieved_indices,
            'reciprocal_rank': rr,
        })

    # Average across queries
    avg_recall_at_k = {k: np.mean(vals) for k, vals in recall_at_k.items()}
    mrr = np.mean(reciprocal_ranks)

    return {
        'recall_at_k': avg_recall_at_k,
        'mrr': mrr,
        'per_query': per_query_results,
    }


# Define our evaluation set with ground truth
# Each tuple is (query, set of indices of relevant passages)
eval_set = [
    ("How do solar cells generate electricity?", {0}),
    ("What is the cost of solar power?", {1}),
    ("How long do solar panels last?", {2}),
    ("What is the maximum efficiency of a wind turbine?", {3}),
    ("Advantages of offshore wind energy", {4}),
    ("How much do lithium batteries cost?", {5}),
    ("Explain pumped hydro storage", {6}),
    ("Global temperature increase and Paris Agreement", {7, 8}),
    ("What is the duck curve?", {9}),
    ("How is green hydrogen produced?", {10}),
    ("What is the capacity factor of different energy sources?", {11}),
    ("Renewable energy storage technologies", {5, 6}),
]

eval_results = evaluate_retrieval(eval_set, max_k=10)

print("Retrieval Evaluation Results")
print("=" * 50)
print(f"Mean Reciprocal Rank (MRR): {eval_results['mrr']:.4f}")
print(f"\nRecall@K:")
for k in [1, 3, 5, 10]:
    print(f"  Recall@{k:<2d} = {eval_results['recall_at_k'][k]:.4f}")

print(f"\nPer-query breakdown:")
for pq in eval_results['per_query']:
    rr_str = f"1/{int(1/pq['reciprocal_rank'])}" if pq['reciprocal_rank'] > 0 else "miss"
    print(f"  \"{pq['query'][:45]}...\"  RR={rr_str}")

In [ ]:
#@title 🎧 What to Look For: Recall Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_29_recall_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualization: Recall@K Curve

This is the standard way to visualize retrieval quality. The curve shows how recall improves as we retrieve more documents.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel 1: Recall@K Curve ---
ax1 = axes[0]
k_values = list(eval_results['recall_at_k'].keys())
recall_values = [eval_results['recall_at_k'][k] for k in k_values]

ax1.plot(k_values, recall_values, 'o-', color='#3B82F6', linewidth=2.5,
         markersize=8, markerfacecolor='white', markeredgewidth=2, zorder=3)

# Fill area under curve
ax1.fill_between(k_values, recall_values, alpha=0.15, color='#3B82F6')

# Mark the "sweet spot" — where recall plateaus
for i in range(1, len(recall_values)):
    if recall_values[i] - recall_values[i-1] < 0.02:  # diminishing returns
        sweet_k = k_values[i]
        ax1.axvline(x=sweet_k, color='#EF4444', linestyle='--', alpha=0.6)
        ax1.annotate(f'Diminishing returns\nafter K={sweet_k}',
                     xy=(sweet_k, recall_values[i]),
                     xytext=(sweet_k + 1.5, recall_values[i] - 0.1),
                     arrowprops=dict(arrowstyle='->', color='#EF4444'),
                     fontsize=10, color='#EF4444', fontweight='bold')
        break

ax1.set_xlabel('K (number of retrieved passages)', fontsize=13)
ax1.set_ylabel('Recall@K', fontsize=13)
ax1.set_title('Recall@K Curve\nHow recall improves with more retrieval',
              fontsize=14, fontweight='bold')
ax1.set_ylim(0, 1.05)
ax1.set_xticks(k_values)

# --- Panel 2: Per-Query Reciprocal Rank ---
ax2 = axes[1]
query_labels = [pq['query'][:30] + '...' for pq in eval_results['per_query']]
rr_values = [pq['reciprocal_rank'] for pq in eval_results['per_query']]

colors = ['#10B981' if rr == 1.0 else '#F59E0B' if rr >= 0.5 else '#EF4444' for rr in rr_values]
bars = ax2.barh(range(len(query_labels)), rr_values, color=colors, alpha=0.8,
                edgecolor='white', height=0.7)
ax2.set_yticks(range(len(query_labels)))
ax2.set_yticklabels(query_labels, fontsize=8)
ax2.set_xlabel('Reciprocal Rank', fontsize=13)
ax2.set_title(f'Per-Query Reciprocal Rank\nMRR = {eval_results["mrr"]:.3f}',
              fontsize=14, fontweight='bold')
ax2.set_xlim(0, 1.1)
ax2.invert_yaxis()

# Add value labels
for bar, rr in zip(bars, rr_values):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{rr:.2f}', va='center', fontsize=9, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#10B981', alpha=0.8, label='RR = 1.0 (rank 1)'),
    Patch(facecolor='#F59E0B', alpha=0.8, label='RR = 0.5 (rank 2)'),
    Patch(facecolor='#EF4444', alpha=0.8, label='RR < 0.5 (rank 3+)'),
]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nKey takeaways:")
print(f"  - Recall@1 = {eval_results['recall_at_k'][1]:.2f} (how often the top result is relevant)")
print(f"  - Recall@3 = {eval_results['recall_at_k'][3]:.2f} (how often a relevant result is in top 3)")
print(f"  - MRR = {eval_results['mrr']:.3f} (on average, the first relevant result is at rank ~{1/eval_results['mrr']:.1f})")

In [ ]:
#@title 🎧 Transition: Final Output Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_30_final_output_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Final Output — Full Pipeline on Diverse Questions

Let us run our complete RAG pipeline on 5 diverse questions to demonstrate its range. For each question, we display the retrieved sources with relevance scores and the assembled prompt.

In [ ]:
#@title 🎧 Code Walkthrough: Final Output Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_31_final_output_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
final_queries = [
    "What is the most cost-effective form of renewable energy right now?",
    "How can we store renewable energy for use at night?",
    "What are the environmental targets set by the Paris Agreement?",
    "Why do solar-heavy grids experience the duck curve problem?",
    "Compare the capacity factors of different energy sources.",
]

print("=" * 70)
print("FINAL RAG PIPELINE OUTPUT — 5 Diverse Questions")
print("=" * 70)

summary_data = []

for i, query in enumerate(final_queries, 1):
    result = rag_pipeline(query, top_k=3, verbose=False)

    print(f"\n{'─'*70}")
    print(f"Question {i}: {query}")
    print(f"{'─'*70}")

    print(f"\nRetrieved Sources:")
    for rank, (text, score, idx) in enumerate(result['retrieved'], 1):
        # Show the full passage with citation
        print(f"\n  [Source {rank}] (relevance: {score:.4f}, Passage {idx})")
        # Wrap long text for readability
        wrapped = textwrap.fill(text, width=70, initial_indent='    ', subsequent_indent='    ')
        print(wrapped)

    summary_data.append({
        'Question': query[:50] + '...' if len(query) > 50 else query,
        'Top Score': f"{result['retrieved'][0][1]:.4f}",
        'Sources': ', '.join([f"P{idx}" for _, _, idx in result['retrieved']]),
        'Prompt Tokens': f"~{result['total_context_chars']//4}",
    })

# Summary table
print(f"\n\n{'='*70}")
print("SUMMARY TABLE")
print(f"{'='*70}")

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

print(f"\nTotal queries processed: {len(final_queries)}")
print(f"Average top-1 relevance score: "
      f"{np.mean([r['retrieved'][0][1] for r in [rag_pipeline(q, verbose=False) for q in final_queries]]):.4f}")

In [ ]:
#@title 🎧 Listen: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_32_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### What Did We Build?

We built a complete RAG system from scratch:

1. **Knowledge Base** — 12 factual passages about renewable energy and climate
2. **Chunking** — Fixed-size splitting with and without overlap
3. **Embedding** — Converted text to 384-dimensional vectors using sentence-transformers
4. **FAISS Index** — Fast similarity search over embedded passages
5. **Retrieval** — Top-K nearest neighbor search using cosine similarity
6. **Prompt Assembly** — Structured prompts with source citations
7. **Evaluation** — Recall@K and MRR metrics to measure retrieval quality

The key insight from the mathematical formulation is worth repeating:

$$P(y \mid x) = \sum_{z} P(y \mid x, z) \cdot P(z \mid x)$$

**The quality of your RAG system depends on two independent factors**: how good your retriever is ($P(z|x)$) and how well your generator uses the retrieved context ($P(y|x,z)$). You can improve either one independently.

### Reflection Questions

1. **The Cold Start Problem**: Our knowledge base had only 12 passages. What changes when you have 12 million? Which components of our pipeline would need to change, and which would stay the same?

2. **The Chunking Dilemma**: We saw that chunk size affects retrieval quality. But in practice, the optimal chunk size depends on the *type* of questions users ask. Short factual questions benefit from small chunks. Complex analytical questions need larger chunks with more context. How would you handle a system that gets both types?

3. **Beyond Text**: Our pipeline only handles text. But real knowledge bases contain tables, images, code, and structured data. How would you extend RAG to handle a PDF with charts and tables?

### Optional Challenges

1. **Implement Hybrid Search**: Combine our embedding-based search with keyword matching (BM25 or TF-IDF). Use a weighted combination: `final_score = alpha * embedding_sim + (1 - alpha) * keyword_score`. Does hybrid search outperform either method alone?

2. **Build a Query Expansion Module**: Before searching, use an LLM to generate 3 variations of the user's query. Retrieve results for all variations, merge and deduplicate. Does this improve recall?

3. **Implement a Context Budget Manager**: Given a token budget (say 2000 tokens), select the maximum number of retrieved passages that fit, prioritizing by relevance score. Handle the case where a single passage exceeds the budget by truncating it intelligently.

---


In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_33_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


You now have a working RAG system and the conceptual tools to understand any production RAG pipeline you encounter. Whether it is LangChain, LlamaIndex, or a custom system — the core mechanics are exactly what we built here: embed, retrieve, assemble, generate.

That understanding is what separates an engineer who *uses* tools from one who *builds* them.

---

*Part 1 of the Vizuara series on RAG Systems*